# Evaluation against the hidden plant (simulation study only)

This notebook exists only because the "real rig" here is a MuJoCo model whose true parameters are known. It compares:
* `identified_parameters.json`, written by `02_parameter_identification.ipynb` from encoder logs only;
* `dataset/ground_truth.json`, written by `mujoco/make_real_plant.py`.

**On the physical rig there is no ground truth.** There, the validation runs in section 7 of notebook 02 are the evidence that the model is right. Nothing from this notebook flows back into the identification.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'identification' else Path.cwd()
FIG = ROOT / 'identification' / 'figures'
FIG.mkdir(exist_ok=True)
ident = json.loads((ROOT / 'identification' / 'identified_parameters.json').read_text())
truth = json.loads((ROOT / 'dataset' / 'ground_truth.json').read_text())

cad = dict(m_p=0.01725, l_p=0.04277, c_p=-0.00177, I_px=1.5613e-5, I_py=1.5204e-5, I_pz=8.93e-7,
           I_pxz=1.309e-6, b_p=1.744e-5, F_p=3.457e-5, r=0.062, g=9.81)
J2c = cad['I_px'] + cad['m_p']*cad['l_p']**2
cad_abs = dict(J2=J2c, G=cad['g']*cad['m_p']*cad['l_p'], K=cad['I_pxz'] + cad['m_p']*cad['l_p']*(cad['r'] + cad['c_p']),
               J1=cad['I_py'] - cad['I_pz'] + cad['m_p']*cad['l_p']**2, b_p=cad['b_p'], F_p=cad['F_p'])
cad_rat = dict(kappa=cad_abs['K']/J2c, rho=cad_abs['J1']/J2c, omega2=cad_abs['G']/J2c, beta=cad_abs['b_p']/J2c, phi=cad_abs['F_p']/J2c)

## Ratios: everything the controller needs

In [ ]:
print(f'{"ratio":7s} {"CAD guess":>11s} {"identified":>11s} {"truth":>11s} {"CAD err":>8s} {"id err":>8s} {"id 1-sigma":>10s}')
rows = []
for k in ['kappa', 'rho', 'omega2', 'beta', 'phi']:
    c, i, t, s = cad_rat[k], ident['ratios'][k], truth['ratios'][k], ident['ratios_1sigma'][k]
    rows.append((k, 100*(c - t)/t, 100*(i - t)/t))
    print(f'{k:7s} {c:11.5g} {i:11.5g} {t:11.5g} {rows[-1][1]:7.1f}% {rows[-1][2]:7.2f}% {100*s/i:9.3f}%')

## Absolute values from the added-mass experiment

In [ ]:
tl = truth['lumped']
for k in ['J2', 'G', 'K', 'J1', 'b_p', 'F_p']:
    c, i, t = cad_abs[k], ident['absolute_from_added_mass'][k], tl[k]
    s = ident['absolute_1sigma'][k]
    rows.append((k, 100*(c - t)/t, 100*(i - t)/t))
    print(f'{k:7s} CAD {c:11.4e}  identified {i:11.4e} +- {100*s/i:4.1f}%   truth {t:11.4e}   CAD err {rows[-1][1]:6.1f}%  id err {rows[-1][2]:6.2f}%')
tp = truth['physical']
if ident['pendulum_com']:
    pc = ident['pendulum_com']
    print(f"\nl_p: CAD {cad['l_p']*1e3:.2f} mm, identified {pc['l_p']*1e3:.2f} +- {pc['l_p_1sigma']*1e3:.2f} mm, truth {tp['l_p']*1e3:.2f} mm")
    print(f"c_p + I_pxz/(m_p l_p): identified {pc['c_p_plus_Ipxz_over_mplp']*1e3:+.2f} mm, "
          f"truth {(tp['c_p'] + tp['I_pxz']/(tp['m_p']*tp['l_p']))*1e3:+.2f} mm  (true c_p alone {tp['c_p']*1e3:+.2f} mm)")
print(f"\nAS5600 zero offset: identified {np.rad2deg(ident['as5600_zero_offset_rad']):.3f} deg, "
      f"truth {np.rad2deg(truth['sensors']['as5600_zero_offset_rad']):.3f} deg")
print('\nNot identified (no torque): J0, b_a, F_a, m_a, x_a. For reference only, their true values:',
      {k: f'{v:.3e}' for k, v in tl.items() if k in ('J0', 'b_a', 'F_a')})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(rows))
ax.bar(x - 0.2, [r[1] for r in rows], 0.38, color='#e09f3e', label='CAD guess')
ax.bar(x + 0.2, [r[2] for r in rows], 0.38, color='#2a6fdb', label='identified (encoders only)')
ax.axhline(0, color='#333333', lw=0.8)
ax.axvline(4.5, color='#999999', lw=0.8, ls=':')
ax.text(2, 38, 'ratios (controller model)', ha='center', fontsize=9, color='#555555')
ax.text(8, 38, 'absolute (added mass)', ha='center', fontsize=9, color='#555555')
ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows])
ax.set_ylim(-45, 45); ax.set_ylabel('error vs. hidden truth [%]')
ax.set_title('Identification error against the hidden MuJoCo plant')
ax.grid(True, axis='y', alpha=0.3); ax.legend(loc='lower right')
plt.tight_layout(); fig.savefig(FIG / 'parameter_errors.png', dpi=110); plt.show()